<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 그래프 학습 · 추천 시스템 · 00. DeepWalk: 그래프를 문장처럼 읽기

## DeepWalk: Online Learning of Social Representations

- **원 논문:** [DeepWalk: Online Learning of Social Representations](https://arxiv.org/abs/1403.6652)
- **저자 / 발표:** Bryan Perozzi, Rami Al-Rfou, Steven Skiena · KDD (2014)
- **난이도 / 예상 시간:** 초급 · 약 60분
- **선수 지식:** 확률, random walk, softmax, 임베딩
- **로컬 학습 데이터:** `data/field_curriculum/graph_recommendation.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 절단 random walk를 문장으로 보고 Skip-gram 문맥쌍을 만든 뒤 작은 그래프의 노드 임베딩을 학습합니다.

**축소하거나 생략한 부분:** BlogCatalog/Flickr/YouTube와 hierarchical softmax 대신 18개 노드, full-softmax를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1, §4.2 및 Algorithm 1 | 각 정점에서 절단 random walk 생성 | 길이·이웃 전이·seed 재현성 |
| §4.2.1 및 Algorithm 2 | walk의 중심-문맥 쌍 생성 | window 경계와 self-pair 제외 |
| §3.3, Eq. (2) 및 Algorithm 2 lines 3–4 | Skip-gram 조건부 우도 최적화 | 초기/최종 cross-entropy |
| Figure 3 | walk→문맥→표현 파이프라인 | 동일 label/상이 label cosine |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** 중심/문맥 id [P] → embedding [P,D] → node logits [P,N]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(7)

DATA_PATH = Path("data/field_curriculum/graph_recommendation.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
adjacency = torch.tensor(raw["adjacency"], dtype=torch.float32)
features = torch.tensor(raw["features"], dtype=torch.float32)
labels = torch.tensor(raw["labels"], dtype=torch.long)
train_mask = torch.tensor(raw["train_mask"], dtype=torch.bool)
test_mask = torch.tensor(raw["test_mask"], dtype=torch.bool)
assert adjacency.ndim == 2 and adjacency.shape[0] == adjacency.shape[1] == len(features)
assert torch.allclose(adjacency, adjacency.T) and not torch.diagonal(adjacency).any()
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={len(features)}, edges={edge_count}, features={features.shape[1]}")
print(ACCELERATOR.summary())
print(
    "graph sampling/topology preprocessing stays on CPU, "
    "while neural training uses DEVICE"
)

## 포트폴리오 해설: 핵심 수식에서 실행 코드까지

$$\max_{\Phi}\sum_{v\in V}\sum_{u\in\mathcal{N}_w(v)}\log p(u\mid v)$$

- **기호 정의:** v는 중심 노드, u는 walk 문맥, Φ는 노드 임베딩, N_w는 window 문맥입니다.
- **수식의 역할:** random walk를 문장으로 바꾼 뒤 Skip-gram 우도를 높여 구조적 근접성을 학습합니다.
- **구현 이유:** 원문 전체를 축약하더라도 위 관계가 결과를 만드는 핵심이므로,
  이 수식을 전처리·`forward`·loss 경계로 나누어 직접 구현했습니다.
- **Task/코드 대응:** Task P의 `preprocess_graph`, 논문 전용 클래스의
  `forward`, `compute_loss`, `training_step`이 §3.1, §4.2 및 Algorithm 1을 구조화합니다.
- **입출력 shape:** 노드 id [P] → 전체 노드 logit [P, N]
- **평가:** cross-entropy 감소와 동일 클래스 노드의 평균 cosine 분리도
- **원문 대비 한계:** BlogCatalog/Flickr/YouTube와 hierarchical softmax 대신 18개 노드, full-softmax를 사용합니다.

코드를 읽을 때는 `Config → 전처리 → Module.forward → loss → train → evaluate`
순서로 추적하세요. 각 함수는 한 가지 책임만 갖도록 분리했습니다.

## 1. Algorithm 1: 절단 random walk

DeepWalk의 핵심 전환은 **정점=단어, walk=문장**입니다. 아래 구현은 현재 정점의 이웃에서만
균등 표본을 뽑습니다. 논문의 병렬/streaming 실행은 생략하지만 표본 과정은 같습니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1, §4.2 및 Algorithm 1
- **이 셀의 책임:** 각 정점에서 절단 random walk 생성
- **Tensor shape 계약:** 중심/문맥 id [P] → embedding [P,D] → node logits [P,N]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 길이·이웃 전이·seed 재현성. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def random_walk(adj: Tensor, start: int, length: int, seed: int) -> list[int]:
    rng = np.random.default_rng(seed)
    walk = [int(start)]
    while len(walk) < length:
        neighbors = torch.where(adj[walk[-1]] > 0)[0].numpy()
        if len(neighbors) == 0:
            break
        walk.append(int(rng.choice(neighbors)))
    return walk


walk = random_walk(adjacency, 0, 12, 11)
assert len(walk) == 12
assert all(adjacency[a, b] == 1 for a, b in zip(walk, walk[1:]))
assert walk == random_walk(adjacency, 0, 12, 11)
print("walk:", walk)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §4.2.1 및 Algorithm 2
- **이 셀의 책임:** walk의 중심-문맥 쌍 생성
- **Tensor shape 계약:** 중심/문맥 id [P] → embedding [P,D] → node logits [P,N]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** window 경계와 self-pair 제외. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def skipgram_pairs(walks: list[list[int]], window: int = 2) -> Tensor:
    pairs = []
    for seq in walks:
        for i, center in enumerate(seq):
            for j in range(max(0, i - window), min(len(seq), i + window + 1)):
                if i != j:
                    pairs.append((center, seq[j]))
    return torch.tensor(pairs, dtype=torch.long)


walks = [
    random_walk(adjacency, s, 12, 100 * s + r)
    for s in range(len(features))
    for r in range(2)
]
pairs = skipgram_pairs(walks)
pairs = pairs[torch.randperm(len(pairs))[: min(2400, len(pairs))]]
assert pairs.ndim == 2 and pairs.shape[1] == 2 and len(pairs) > len(walks)
print("walks / positive pairs:", len(walks), len(pairs))

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3.3, Eq. (2) 및 Algorithm 2 lines 3–4
- **이 셀의 책임:** Skip-gram 조건부 우도 최적화
- **Tensor shape 계약:** 중심/문맥 id [P] → embedding [P,D] → node logits [P,N]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 초기/최종 cross-entropy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
n, dim = len(features), 8
pairs_device = ACCELERATOR.move(pairs)
input_embed = ACCELERATOR.move(nn.Embedding(n, dim))
output_embed = ACCELERATOR.move(nn.Embedding(n, dim))
optimizer = torch.optim.Adam(
    [*input_embed.parameters(), *output_embed.parameters()], lr=0.04
)
losses = []
for step in range(55):
    logits = input_embed(pairs_device[:, 0]) @ output_embed.weight.T
    loss = F.cross_entropy(logits, pairs_device[:, 1])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
node_embedding = F.normalize(input_embed.weight.detach(), dim=1).cpu()
assert losses[-1] < losses[0] * 0.9 and torch.isfinite(node_embedding).all()
print(f"Skip-gram CE: {losses[0]:.3f} → {losses[-1]:.3f}")

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** Figure 3
- **이 셀의 책임:** walk→문맥→표현 파이프라인
- **Tensor shape 계약:** 중심/문맥 id [P] → embedding [P,D] → node logits [P,N]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 동일 label/상이 label cosine. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
similarity = node_embedding @ node_embedding.T
eye = torch.eye(len(labels), dtype=torch.bool)
same = (labels[:, None] == labels[None, :]) & ~eye
different = labels[:, None] != labels[None, :]
same_mean = float(similarity[same].mean())
diff_mean = float(similarity[different].mean())
assert same_mean > diff_mean
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set(title="Skip-gram loss", xlabel="step")
ax[1].bar(["same", "different"], [same_mean, diff_mean])
ax[1].set(title="cosine by label")
plt.tight_layout()
plt.show()
print({"same_cosine": round(same_mean, 3), "different_cosine": round(diff_mean, 3)})

### Task P1 · 전처리와 핵심 모델

먼저 graph 입력의 shape와 정규화 규칙을 고정하고, 논문의 핵심 연산을
`nn.Module`의 `forward`와 `compute_loss`로 분리합니다.

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** §3.1, §4.2 및 Algorithm 1 / §4.2.1 및 Algorithm 2 / §3.3, Eq. (2) 및 Algorithm 2 lines 3–4
- **이 셀의 책임:** 각 정점에서 절단 random walk 생성 / walk의 중심-문맥 쌍 생성 / Skip-gram 조건부 우도 최적화
- **Tensor shape 계약:** 중심/문맥 id [P] → embedding [P,D] → node logits [P,N]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 길이·이웃 전이·seed 재현성 / window 경계와 self-pair 제외 / 초기/최종 cross-entropy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    embedding_dim: int = 10
    walks_per_node: int = 2
    walk_length: int = 10
    context_window: int = 2
    max_pairs: int = 720
    learning_rate: float = 0.04
    steps: int = 16


def preprocess_graph(graph: Tensor, config: PortfolioConfig) -> Tensor:
    walks = [
        random_walk(
            graph,
            start=node_id,
            length=config.walk_length,
            seed=1_000 * node_id + walk_id,
        )
        for node_id in range(len(graph))
        for walk_id in range(config.walks_per_node)
    ]
    pairs = skipgram_pairs(walks, window=config.context_window)
    generator = torch.Generator().manual_seed(7)
    order = torch.randperm(len(pairs), generator=generator)
    return pairs[order[: config.max_pairs]]


class DeepwalkPortfolioModel(nn.Module):
    def __init__(self, node_count: int, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.center = nn.Embedding(node_count, config.embedding_dim)
        self.context = nn.Embedding(node_count, config.embedding_dim)

    def forward(self, node_ids: Tensor) -> Tensor:
        return self.center(node_ids) @ self.context.weight.T

    def compute_loss(self, pairs: Tensor) -> Tensor:
        return F.cross_entropy(self(pairs[:, 0]), pairs[:, 1])

    def training_step(self, pairs: Tensor) -> Tensor:
        return self.compute_loss(pairs)

### Task P2 · objective와 update

`fit_portfolio_model`은 gradient가 흐르는 parameter와 한 step의 순서를
드러냅니다. 논문의 목적함수가 코드의 어느 tensor에 적용되는지 확인하세요.

### 구현 단계 7 · 목적함수·학습 update

- **원문 위치:** §3.3, Eq. (2) 및 Algorithm 2 lines 3–4
- **이 셀의 책임:** Skip-gram 조건부 우도 최적화
- **Tensor shape 계약:** 중심/문맥 id [P] → embedding [P,D] → node logits [P,N]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 초기/최종 cross-entropy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: DeepwalkPortfolioModel,
    pairs: Tensor,
) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(pairs)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 평가와 한계 기록

loss와 별개의 논문 지표를 계산하고, 로컬 합성 graph에서 얻은 수치가
원 논문의 대규모 benchmark 결과를 대신하지 않는다는 점을 기록합니다.

### 구현 단계 8 · 평가·완료 증거

- **원문 위치:** Figure 3
- **이 셀의 책임:** walk→문맥→표현 파이프라인
- **Tensor shape 계약:** 중심/문맥 id [P] → embedding [P,D] → node logits [P,N]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 동일 label/상이 label cosine. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(model: DeepwalkPortfolioModel) -> dict[str, float]:
    embedding = F.normalize(model.center.weight.detach(), dim=1)
    similarity = embedding @ embedding.T
    same = labels[:, None] == labels[None, :]
    eye = torch.eye(len(labels), dtype=torch.bool)
    separation = similarity.cpu()[same & ~eye].mean() - similarity.cpu()[~same].mean()
    return {"class_cosine_gap": float(separation)}


portfolio_config = PortfolioConfig()
portfolio_pairs = ACCELERATOR.move(preprocess_graph(adjacency, portfolio_config))
portfolio_model = ACCELERATOR.move(
    DeepwalkPortfolioModel(len(features), portfolio_config)
)
portfolio_history = fit_portfolio_model(portfolio_model, portfolio_pairs)
portfolio_metrics = evaluate_portfolio_model(portfolio_model)
assert len(portfolio_pairs) > len(features)
assert np.isfinite(portfolio_history).all()
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> BlogCatalog/Flickr/YouTube와 hierarchical softmax 대신 18개 노드, full-softmax를 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.